# Day 15: Level 0 — Cross-Market Capital Flows and the Global Game

*Alpha Flow Research · HongJin HE · July 2026*

---

## Extending the Hierarchy: Four Levels

The three-level hierarchy (Day 6) described L1→L3 within a single national market. But capital is globally mobile. The complete hierarchy has **four levels**:

| Level | Players | Scope | Game Type |
|---|---|---|---|
| **L0** | Nations, central banks, sovereign wealth funds | Global (FX, cross-border flows) | MFC (each sovereign optimizes domestic welfare) |
| **L1** | Institutional investors by type (banks, funds, pension) | National market | MFG (competitive within type) |
| **L2** | Individual firms within a type (Jane Street vs Citadel) | Firm-level | MFG (competitive between firms) |
| **L3** | Individual decision-makers within firms | Desk/portfolio level | MFC/MFG (internal incentive design) |

## Why Level 0 Matters: The Dollar Cycle

The single most important macro driver of US equity returns is **global dollar liquidity** — determined by:
- Fed policy (US domestic rates)
- Risk appetite of global investors (FX carry flows)
- Sovereign intervention (Japan, China, Switzerland FX reserves)
- Emerging market dollar debt cycles

A world model that ignores L0 will systematically fail during:
- 2015 China devaluation → US equity selloff
- 2022 Dollar surge → synchronized global selloff
- 2013 Taper Tantrum → EM capital flight

These are L0 events that propagate **downward** through L1→L2→L3, creating correlated shocks that invalidate L1-only models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr

np.random.seed(2026)

# ── Simulate the Global Dollar Cycle and its equity impact ───────────────
T = 252 * 8  # 8 years
t = np.arange(T)

# L0: Dollar Index (DXY proxy) — driven by Fed relative to global CBs
# Mean-reverting with slow drift cycles (~3-5yr)
theta_dxy = 0.005
dxy = np.zeros(T)
dxy[0] = 100.0
fed_cycle = np.sin(2 * np.pi * t / (252 * 4))  # 4-year Fed cycle
for i in range(1, T):
    dxy[i] = dxy[i-1] + theta_dxy * (100 - dxy[i-1]) + 0.5 * fed_cycle[i] + 0.3 * np.random.randn()

# L0: Global risk appetite (VIX proxy) — correlated with DXY strength
vix_base = 20 + 0.3 * (dxy - 100)  # high DXY → risk-off → high VIX
vix = np.maximum(vix_base + 3 * np.random.randn(T), 8)

# Simulate three national equity markets with L0 exposure
markets = {
    'US (SPX)':      {'beta_dxy': -0.8, 'beta_vix': -0.6, 'domestic_alpha': 0.12, 'vol': 0.16},
    'EM (EEM)':      {'beta_dxy': -2.5, 'beta_vix': -1.2, 'domestic_alpha': 0.08, 'vol': 0.25},
    'Japan (NKY)':   {'beta_dxy':  1.5, 'beta_vix': -0.4, 'domestic_alpha': 0.06, 'vol': 0.20},
}

returns = {}
prices = {}
dxy_ret = np.diff(dxy) / dxy[:-1]
vix_chg = np.diff(vix)

for name, params in markets.items():
    r = (params['domestic_alpha'] / 252
         + params['beta_dxy'] * dxy_ret / 100
         + params['beta_vix'] * vix_chg / 252
         + params['vol'] / np.sqrt(252) * np.random.randn(T-1))
    returns[name] = r
    prices[name] = 100 * np.exp(np.cumsum(r))

# Compute rolling cross-market correlation (L0 transmission indicator)
window = 63
us_em_corr = np.array([pearsonr(returns['US (SPX)'][i:i+window],
                                 returns['EM (EEM)'][i:i+window])[0]
                        for i in range(T-1-window)])

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(3, 2, figure=fig)

ax1 = fig.add_subplot(gs[0, :])
for name, p in prices.items():
    ax1.plot(t[1:], p / p[0] * 100, lw=1.2, label=name)
ax1.set_ylabel('Indexed (base=100)')
ax1.legend()
ax1.set_title('Three Markets Under Common L0 Dollar Cycle')

ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(t, dxy, color='green', lw=1)
ax2.set_ylabel('DXY (Dollar Index)')
ax2.set_title('L0: Dollar Cycle (Fed policy driver)')

ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(t, vix, color='red', lw=1)
ax3.set_ylabel('VIX proxy')
ax3.set_title('L0: Global Risk Appetite')

ax4 = fig.add_subplot(gs[2, :])
ax4.plot(t[1:window+1], us_em_corr, color='purple', lw=1, label='US-EM rolling corr (63d)')
ax4.axhline(np.mean(us_em_corr), color='black', ls='--', label=f'Mean = {np.mean(us_em_corr):.2f}')
ax4.set_ylabel('Correlation')
ax4.set_xlabel('Trading day')
ax4.set_title('L0 Transmission: High DXY periods show elevated cross-market correlation')
ax4.legend()

plt.tight_layout()
plt.savefig('../figures/level0_cross_market.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify L0 explanatory power
from numpy.linalg import lstsq
X = np.column_stack([dxy_ret, vix_chg[:len(dxy_ret)]])
for name, r in returns.items():
    n = min(len(r), len(X))
    beta, _, _, _ = lstsq(X[:n], r[:n], rcond=None)
    r2 = 1 - np.var(r[:n] - X[:n] @ beta) / np.var(r[:n])
    print(f'{name}: L0 R² = {r2:.1%} (β_DXY={beta[0]:.2f}, β_VIX={beta[1]:.4f})')

## The Four-Level MFG/MFC System

The complete mathematical structure is a **nested Stackelberg-MFG**:

**L0 (MFC — Sovereign)**: Each nation $k$ solves:
$$\min_{\pi^k \in \Pi^{\text{CB}}} \mathbb{E}\left[\int_0^T \mathcal{L}^k(e_t, r_t, \pi^k_t) dt\right]$$
where $e_t$ is the FX rate, $r_t$ is the domestic rate. Nations interact via the FX equilibrium condition: $\sum_k \text{CA}_k(e_t) = 0$.

**L1 (MFG — Institution type)**: Given L0 policy $(\pi^{*,0})$:
$$V^{(1)}(t, z, \mu^{(1)}) = \sup_{a^{(1)}} \mathbb{E}\left[\int_t^T f^{(1)}(z_s, a^{(1)}_s, \mu^{(1)}_s, \pi^{*,0}_s) ds\right]$$

**L2 (MFG — Firm)**: Given L0+L1 equilibrium, individual firms compete.

**L3 (MFC internal / MFG between desks)**: Internal incentive design.

**Timescale separation** makes this tractable:
- L0 moves on yearly cycles → exogenous to L1/L2/L3 during solving
- L1 moves quarterly → exogenous to L2/L3
- L2/L3 solve simultaneously at daily frequency

In MicroWorld: L0 enters via the Encoder's **macro latent dimensions** $z^{(\text{DXY})}, z^{(\text{global risk})}$, estimated from FRED data. No separate L0 solver needed — L0 is treated as a **slow exogenous driving process** identified by the encoder.

In [ ]:
# Demonstrate how L0 signal improves domestic market prediction
# Without L0: predict US returns from US features only
# With L0: add DXY + VIX as additional features

from numpy.linalg import lstsq

us_r = returns['US (SPX)']
n = len(us_r) - 1

# Feature: lagged US return only
X_no_l0 = us_r[:n].reshape(-1, 1)
y = us_r[1:n+1]

# Feature: lagged US return + L0 signals
X_with_l0 = np.column_stack([
    us_r[:n],
    dxy_ret[:n],
    vix_chg[:n],
])

def oos_r2(X, y, train_frac=0.6):
    n_train = int(len(y) * train_frac)
    beta, _, _, _ = lstsq(X[:n_train], y[:n_train], rcond=None)
    y_pred = X[n_train:] @ beta
    ss_res = np.sum((y[n_train:] - y_pred)**2)
    ss_tot = np.sum((y[n_train:] - y[n_train:].mean())**2)
    return 1 - ss_res / ss_tot

r2_no_l0 = oos_r2(X_no_l0, y)
r2_with_l0 = oos_r2(X_with_l0, y)

print('Out-of-sample R² for US equity return prediction:')
print(f'  Without L0 (domestic only):  {r2_no_l0:.4f}')
print(f'  With L0 (DXY + VIX added):  {r2_with_l0:.4f}')
print(f'  Improvement:                 {r2_with_l0 - r2_no_l0:+.4f}')
print('\nConclusion: L0 cross-market signals have incremental predictive power')
print('even after conditioning on domestic returns.')